In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


### 1. Data Collection/Preparation

Now that the dataset is downloaded, let's load it into a pandas DataFrame.

In [2]:
import pandas as pd
import os

# The 'path' variable from the previous cell points to the dataset directory.
# Let's find the CSV file within that directory.

# List contents of the downloaded directory to find the CSV file
dataset_files = os.listdir(path)
csv_file = [f for f in dataset_files if f.endswith('.csv')][0]

# Construct the full path to the CSV file
full_csv_path = os.path.join(path, csv_file)

# Load the dataset into a pandas DataFrame
imdb_df = pd.read_csv(full_csv_path)

# Display the first 5 rows and information about the DataFrame
display(imdb_df.head())
imdb_df.info()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


### 2. Data Preprocessing

This step involves cleaning the text data. For this dataset, we'll focus on removing HTML tags (like `<br />`), converting text to lowercase, removing punctuation, and removing common English stop words. We'll use `nltk` for tokenization and stop word removal.

In [ ]:
import re
import nltk
from nltk.corpus import stopwords

# Download stopwords if needed
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords")

# English stopwords
stop_words = set(stopwords.words("english"))

# Keep words that are important for sentiment/negation
important_words = {
    "not", "no", "nor", "never",
    "very", "too", "really", "quite"
}

# Remove important words from stopwords
stop_words = stop_words - important_words


# Contractions
CONTRACTIONS = {
    "don't": "do not",
    "doesn't": "does not",
    "didn't": "did not",
    "can't": "can not",
    "couldn't": "could not",
    "won't": "will not",
    "wouldn't": "would not",
    "shouldn't": "should not",
    "isn't": "is not",
    "aren't": "are not",
    "wasn't": "was not",
    "weren't": "were not",
    "haven't": "have not",
    "hasn't": "has not",
    "hadn't": "had not",

    "i'm": "i am",
    "you're": "you are",
    "he's": "he is",
    "she's": "she is",
    "it's": "it is",
    "we're": "we are",
    "they're": "they are",

    "i've": "i have",
    "you've": "you have",
    "we've": "we have",
    "they've": "they have",

    "i'll": "i will",
    "you'll": "you will",
    "he'll": "he will",
    "she'll": "she will",
    "we'll": "we will",
    "they'll": "they will",

    "i'd": "i would",
    "you'd": "you would",
    "he'd": "he would",
    "she'd": "she would",
    "we'd": "we would",
    "they'd": "they would"
}


def preprocess_text(text):

    if not isinstance(text, str):
        return ""

    # Remove HTML
    text = re.sub(r"<.*?>", " ", text)

    # Lowercase
    text = text.lower()

    # Expand contractions
    for contraction, expanded in CONTRACTIONS.items():
        text = text.replace(contraction, expanded)

    # Keep letters and spaces
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    words = text.split()

    processed_words = []

    negation_words = {"not", "no", "nor", "never"}

    i = 0

    while i < len(words):

        word = words[i]

        # Handle negation + next word as one feature
        if word in negation_words and i + 1 < len(words):

            next_word = words[i + 1]

            # Don't remove the next word if it's important
            if next_word not in stop_words:

                processed_words.append(
                    word + "_" + next_word
                )

                i += 2
                continue

        # Remove normal stopwords
        if word not in stop_words:
            processed_words.append(word)

        i += 1

    return " ".join(processed_words)


# Apply preprocessing
print("Applying improved text preprocessing...")

imdb_df["processed_review"] = imdb_df["review"].apply(preprocess_text)

# Display examples
display(
    imdb_df[["review", "processed_review", "sentiment"]].head(10)
)

### 3. Feature Extraction

Now, we will convert our preprocessed text data (`processed_review`) into numerical features using TF-IDF (Term Frequency-Inverse Document Frequency). TF-IDF reflects how important a word is to a document in a collection or corpus.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer


# ---------------------------------------------------------
# 1. Prepare text and labels
# ---------------------------------------------------------

texts = imdb_df["processed_review"]

# Positive = 1
# Negative = 0
y = imdb_df["sentiment"].apply(
    lambda x: 1 if x == "positive" else 0
)


# ---------------------------------------------------------
# 2. Split BEFORE fitting TF-IDF
# ---------------------------------------------------------

X_text_train, X_text_test, y_train, y_test = train_test_split(
    texts,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


print("Training reviews:", len(X_text_train))
print("Testing reviews :", len(X_text_test))


# ---------------------------------------------------------
# 3. TF-IDF
# ---------------------------------------------------------

tfidf_vectorizer = TfidfVectorizer(
    max_features=30000,

    # Learn:
    # unigram  -> "bad"
    # bigram   -> "not good"
    # trigram  -> "not very good"
    ngram_range=(1, 3),

    # Ignore extremely rare terms
    min_df=2,

    # Reduce the effect of repeated words
    sublinear_tf=True
)


# IMPORTANT:
# Fit only on training data
X_train = tfidf_vectorizer.fit_transform(X_text_train)

# Transform test data using the already-fitted vectorizer
X_test = tfidf_vectorizer.transform(X_text_test)


print("\nTF-IDF completed.")
print("Shape of X_train:", X_train.shape)
print("Shape of X_test :", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test :", y_test.shape)

print("\nExample features:")
print(tfidf_vectorizer.get_feature_names_out()[:30])

### 4. Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression


model = LogisticRegression(
    C=1,
    solver="liblinear",
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

print("Model trained successfully!")

### 5. Model Evaluation

Now we will evaluate the trained model's performance on the test dataset using various metrics like accuracy, precision, recall, and F1-score. We'll also generate a classification report for a more detailed view.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted') # 'weighted' handles class imbalance
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Negative", "Positive"]
    )
)

### Confusion Matrix

To further understand the model's predictions, let's visualize the confusion matrix. This will show us where the model is making correct and incorrect classifications.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate the confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Negative (0)', 'Positive (1)'],
            yticklabels=['Negative (0)', 'Positive (1)'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

##Testing the Model on Custom Reviews

In [ ]:
sample_reviews = [
    "I love this movie",
    "I hate this movie",
    "I don't like this movie",
    "I do not like this movie",
    "I really don't like this movie",
    "This movie is terrible",
    "This movie is amazing",
    "This movie is fantastic",
    "This movie is boring and terrible",
    "This movie is not good",
    "This movie is not very good",
    "This movie is absolutely wonderful",
    "This movie is not bad"
]


for review in sample_reviews:

    # Preprocess
    processed = preprocess_text(review)

    # TF-IDF
    vector = tfidf_vectorizer.transform([processed])

    # Prediction
    prediction = model.predict(vector)[0]

    # Probability
    probability = model.predict_proba(vector)[0]

    sentiment = "Positive" if prediction == 1 else "Negative"

    print("Review:", review)
    print("Processed:", processed)
    print("Prediction:", sentiment)
    print(f"Positive: {probability[1] * 100:.2f}%")
    print(f"Negative: {probability[0] * 100:.2f}%")
    print("-" * 60)

### 6. Hyperparameter Tuning

Hyperparameter tuning is crucial for optimizing model performance. We'll use `GridSearchCV` to systematically search for the best combination of hyperparameters for our Logistic Regression model.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid to search
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],  # Inverse of regularization strength
    'solver': ['liblinear', 'saga'] # Solvers that support L1 regularization
}

# Initialize Logistic Regression model
log_reg = LogisticRegression(random_state=42, max_iter=1000)

# Initialize GridSearchCV
# We use 'f1_weighted' as scoring metric to account for potential class imbalance (though not significant here)
grid_search = GridSearchCV(estimator=log_reg, param_grid=param_grid,
                           cv=5, scoring='f1_weighted', n_jobs=-1, verbose=1)

print("Starting GridSearchCV...")
# Fit GridSearchCV to the training data
grid_search.fit(X_train, y_train)

print("GridSearchCV completed.")

# Display the best parameters and best score
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best F1-Weighted Score: {grid_search.best_score_:.4f}")

### 7. Evaluate Model with Best Hyperparameters

Now, let's use the model with the best hyperparameters found by `GridSearchCV` to make predictions on the test set and evaluate its performance.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# Get the best model from GridSearchCV
best_model = grid_search.best_estimator_

# Make predictions on the test set with the best model
y_pred_tuned = best_model.predict(X_test)

# Evaluate the tuned model
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
print(f"Accuracy with tuned model: {accuracy_tuned:.4f}")

print(
    classification_report(
        y_test,
        y_pred_tuned,
        target_names=["Negative", "Positive"]
    )
)

In [ ]:
import joblib

joblib.dump(best_model, "sentiment_model.pkl")
joblib.dump(tfidf_vectorizer, "tfidf_vectorizer.pkl")

print("Model saved successfully!")

In [ ]:
from google.colab import files

files.download("sentiment_model.pkl")
files.download("tfidf_vectorizer.pkl")

In [ ]:
import joblib

# Load the saved model
loaded_model = joblib.load("sentiment_model.pkl")
loaded_vectorizer = joblib.load("tfidf_vectorizer.pkl")

### Analyzing Feature Importance

Let's inspect the coefficients of the Logistic Regression model to understand which features (words/n-grams) it considers most important for predicting positive and negative sentiments.

In [ ]:
import numpy as np

# Get feature names from the TF-IDF vectorizer
feature_names = tfidf_vectorizer.get_feature_names_out()

# Get coefficients from the best trained Logistic Regression model
# The coefficients represent the importance of each feature in predicting the target variable.
# For binary classification, positive coefficients contribute to the 'positive' class (1), and negative to the 'negative' class (0).
coefficients = best_model.coef_[0]

# Create a DataFrame to easily view feature names and their coefficients
feature_importance_df = pd.DataFrame({'feature': feature_names, 'coefficient': coefficients})

# Sort features by their coefficients to find the most positive and most negative
feature_importance_df = feature_importance_df.sort_values(by='coefficient', ascending=False)

print("Top 20 Most Positive Features:")
display(feature_importance_df.head(20))

print("\nTop 20 Most Negative Features:")
display(feature_importance_df.tail(20))

### Inspecting Specific Problematic Features

In [ ]:
# Check the coefficients for specific words that were part of misclassified reviews
problematic_features = ['hate', 'like_NEG', 'not_NEG', 'dont like', 'do not like', 'no like', 'not good', 'not bad']

print("Coefficients for specific problematic features:")
for feature in problematic_features:
    if feature in feature_importance_df['feature'].values:
        coefficient = feature_importance_df[feature_importance_df['feature'] == feature]['coefficient'].values[0]
        print(f"- '{feature}': {coefficient:.4f}")
    else:
        print(f"- '{feature}': Not found in model features")

print("\nThis shows how much weight the model is giving to these specific terms. A small or positive coefficient for typically negative words indicates a potential reason for misclassification.")

In [ ]:
test_reviews = [
    "I love this movie",
    "I hate this movie",
    "I don't like this movie",
    "I do not like this movie",
    "This movie is terrible",
    "This movie is amazing",
    "This movie is fantastic",
    "This movie is boring and terrible"
]

for review in test_reviews:

    processed = preprocess_text(review)

    vector = tfidf_vectorizer.transform([processed])

    prediction = model.predict(vector)[0]

    probability = model.predict_proba(vector)[0]

    sentiment = "Positive" if prediction == 1 else "Negative"

    print("Review:", review)
    print("Processed:", processed)
    print("Prediction:", sentiment)
    print(
        f"Positive: {probability[1] * 100:.2f}%"
    )
    print(
        f"Negative: {probability[0] * 100:.2f}%"
    )
    print("-" * 50)

In [ ]:
features_to_check = [
    "like",
    "not",
    "not like",
    "like movie",
    "not like movie",
    "hate",
    "love",
    "terrible",
    "good",
    "not good"
]

feature_names = tfidf_vectorizer.get_feature_names_out()
coefficients = model.coef_[0]

feature_coefficients = dict(
    zip(feature_names, coefficients)
)

print("Feature coefficients:\n")

for feature in features_to_check:

    if feature in feature_coefficients:
        print(
            f"{feature:20s}: "
            f"{feature_coefficients[feature]:+.4f}"
        )
    else:
        print(
            f"{feature:20s}: NOT FOUND"
        )